# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">10/2(금) 오후 · 트리거 · 멱등성 · 스케줄러 — 실습</mark>

오전에 **웹훅 수신 서버**를 만들었습니다. 오후에는 **그 서버로 alert(경보)를 자동으로 전송하는 프로그램**을 만듭니다.

「어제 새벽 3시에 admin 을 노린 로그인이 또 있었는데, 우리 프로그램은 뭘 하고 있었지?」
— 아무것도 안 했습니다. **사람이 실행 버튼을 눌러야만** 돌기 때문입니다.

오늘 오후의 도착점은 **`scheduler_job.py`** 입니다. 사람이 실행하지 않아도 정해진 간격마다 실행됩니다.


## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기</mark>

### 0.1 맨 먼저 · 노트북과 터미널의 위치 맞추기

이 노트북은 `security-agent-toolkit` 의 **`agent_core` 폴더**에 있어야 합니다. 터미널도 같은 폴더에 둡니다.

```bash
cd ~/security-agent-toolkit/agent_core
ls
```

💻 표시가 있는 칸은 노트북 셀이 아니라 **터미널에 입력하는 명령어**입니다.

### 0.2 오늘 오후의 순서

| 교시 | 무엇 |
|---|---|
| 5교시 | 트리거·조건·액션 — 지금까지 만든 게 사실 한 모양이었다 |
| 6교시 | 멱등성 — 같은 알림이 세 번 가면 아무도 안 본다 |
| 7교시 | `schedule` · `cron` · `scheduler_job.py` 조립 |

### 0.3 준비

아래 셀들을 차례로 실행합니다. 오후 실습용 웹훅 수신 서버(`webhook_server_pm.py`)와 9/29 결과 파일을 여기서 만듭니다.
오전에 만든 `webhook_server.py` 는 건드리지 않습니다. 두 파일의 코드는 같습니다.


In [ ]:
# 아침 과제에서 설치한 패키지를 확인합니다
import flask
import requests
import schedule

print("준비 완료")

In [ ]:
%%writefile raw_logs.txt
2026-09-29 09:02:11 INFO accepted login for kim.cs from 10.1.2.11
2026-09-29 09:12:00 WARN failed login for kim01 from 203.0.113.5
2026-09-29 09:15:31 INFO accepted login for lee.yh from 10.1.2.34
2026-09-29 09:16:02 INFO session closed for 10.1.2.34
2026-09-29 10:03:19 WARN failed login for park.js from 10.1.3.7
2026-09-29 10:03:31 INFO accepted login for park.js from 10.1.3.7
2026-09-29 11:20:55 INFO accepted login for choi.mk from 10.1.4.2
2026-09-29 12:40:12 INFO session closed for 10.1.4.2
2026-09-29 03:11:05 WARN failed login for admin from 211.45.12.9
2026-09-29 03:12:47 WARN failed login for admin from 211.45.12.9
2026-09-29 03:13:58 WARN failed login for admin from 211.45.12.9
2026-09-29 03:15:22 WARN failed login for admin from 211.45.12.9
2026-09-29 03:17:09 INFO accepted login for admin from 211.45.12.9
2026-09-29 14:05:38 INFO accepted login for jung.hw from 10.1.2.88
2026-09-29 22:14:03 WARN failed login for kim.cs from 185.220.101.34
2026-09-29 22:14:21 WARN failed login for lee.yh from 185.220.101.34
2026-09-29 22:14:40 WARN failed login for choi.mk from 185.220.101.34
2026-09-29 22:14:58 WARN failed login for jung.hw from 185.220.101.34
2026-09-29 22:15:12 INFO session closed for 185.220.101.34
2026-09-29 23:40:07 WARN failed login for invalid user guest from 185.220.101.34


In [ ]:
import re
import json

PATTERN = r"(?P<time>\d{2}:\d{2}:\d{2}) (?P<level>\w+) \w+ login for (?P<user>[\w.]+) from (?P<ip>[\d.]+)"

rows = []
with open("raw_logs.txt", encoding="utf-8") as f:
    for line in f:
        m = re.search(PATTERN, line)
        if m:
            rows.append(m.groupdict())

with open("normalized_logs.json", "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)

print(f"정규화 {len(rows)}건 — 9/29 결과를 다시 만들었습니다")


In [ ]:
%%writefile webhook_server_pm.py
import argparse
import json
from flask import Flask, request

parser = argparse.ArgumentParser(description="경보를 받는 웹훅 서버")
parser.add_argument("--port", type=int, default=5000)
args = parser.parse_args()

app = Flask(__name__)
received = []


@app.route("/webhook", methods=["POST"])
def webhook():
    event = request.get_json()
    received.append(event)
    with open("received_alerts.json", "w", encoding="utf-8") as f:
        json.dump(received, f, ensure_ascii=False, indent=2)
    return {"status": "ok", "count": len(received)}, 200


app.run(port=args.port)


In [ ]:
import subprocess
import sys
import time


def start_server(path, port):
    """서버 파일을 백그라운드로 실행합니다. 셀이 멈추지 않습니다."""
    proc = subprocess.Popen([sys.executable, path, "--port", str(port)])
    time.sleep(2)
    print(f"{path} 를 {port} 번 포트에서 실행했습니다")
    return proc


server = start_server("webhook_server_pm.py", 5005)


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">5교시 (14:00–14:50) · 트리거 · 조건 · 액션</mark>


## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다. 검색이든 공식 문서든 AI 든 상관없습니다. 찾은 것은 노트북 셀이나 터미널에서 한 번 실행해 확인합니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **워크플로(workflow)** | 일이 흘러가는 순서를 왜 정해 두나 |
| **트리거(trigger)** | 일을 시작하게 만드는 것은 무엇인가 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다. 한 줄씩이면 됩니다.

- 워크플로(workflow) →
- 트리거(trigger) →

적은 내용은 노트북 파일에 함께 저장됩니다.

적어 둔 것은 이 교시가 끝날 때 다시 봅니다. 찾은 것과 배운 것이 어디서 갈렸는지가 오늘의 공부입니다.


---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">지금까지 만든 게 사실 한 모양이었다</mark>


### 왜 필요한가

1. 나흘 동안 만든 것을 늘어놓으면 겉모습이 다 다릅니다. 파일을 읽는 것, API 를 부르는 것, 웹훅을 받는 것.
2. 그런데 **세 조각으로 쪼개 보면 전부 같은 모양**입니다.
3. **무엇이 시작을 알렸는가**(트리거) · **어떤 경우에만 움직이는가**(조건) · **그래서 무엇을 하는가**(액션).


### 나흘치를 한 표에 놓으면

| 트리거 | 조건 | 액션 | 언제 만들었나 |
|---|---|---|---|
| 파일이 있다 | 깨진 줄인가 | 건너뛰고 기록 | 9/28 `log_parser.py` |
| 로그가 쌓였다 | 실패 3회 이상인가 | alert(경보)를 출력한다 | 9/29 룰 ① |
| alert 가 났다 | 처음 보는 IP 인가 | 조회한다 | 9/30 `api_client.py` |
| **시각이 됐다** | **처음 보는 사건인가** | **알린다** | **오늘** |

거창한 것이 아닙니다. **「이런 모양이면 이렇게 하라」**를 코드로 적은 것뿐입니다.


### 이 시간에 나오는 말

| 말 | 뜻 |
|---|---|
| 워크플로 | 일이 흘러가는 정해진 순서 |
| 트리거 | 일을 시작하게 만드는 사건 |
| 조건 | 그중 어떤 경우에만 움직일지 |
| 액션 | 그래서 실제로 하는 일 |


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.1 세 조각으로 갈라 적는다</mark>

9/29에 쓴 룰 ① 코드를 세 조각으로 갈라 보면 이렇습니다.

```python
for row in rows:                          # ← 트리거 : 로그가 있다
    if row["level"] == "WARN":            # ← 조건 : 실패인가
        count[row["user"]] = ...          # ← 액션 : 센다
```

갈라 적으면 **바꿀 곳이 분명해집니다.** 기준을 바꾸려면 조건만, 알림 방식을 바꾸려면 액션만 고칩니다.


아래 셀을 먼저 실행합니다. 오늘 오후 내내 이 `events` 를 씁니다. 9/29 룰 세 개가 찾아낸 alert(경보)입니다.


In [ ]:
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

print(f"경보 {len(events)}건")


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-1 · 무엇이 보일까요</font></h3></td></tr></table>

조건이 하나 붙은 코드입니다. 화면에 무엇이 보일지 적어 보세요.

```python
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

for event in events:
    if event["rule"] == "night_login":
        print(event["time"])
```

막히면 바로 위 `5.1 세 조각으로 갈라 적는다` 설명을 다시 봅니다.


In [ ]:
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

for event in events:
    if event["rule"] == "night_login":
        print(event["time"])


✅ `03:17:09`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-2 · 조건을 하나 붙이기</font></h3></td></tr></table>

`night_login` **이 아닌** alert(경보)만 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `brute_force` · `password_spraying` 두 줄 |

**💡 힌트**

1. 같지 않은지는 `!=` 로 견줍니다.
2. `if` 를 반복 안에 둡니다.
3. 이 줄이 **조건** 조각입니다.


In [ ]:
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

#for event in events:
 # 2. event 의 rule 값이 "night_login" 이 아니면 (!=)
 # 3. 그 rule 값을 출력하세요
 for alert in alerts:
    if alert["rule"] != "night_login":
        print(alert["rule"])



<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-3 · 세 조각에 주석 달기</font></h3></td></tr></table>

아래 코드에서 **트리거·조건·액션**이 어느 줄인지 주석으로 표시하시오.

```python
for event in events:
    if event["rule"] == "brute_force":
        print("[알림]", event["user"])
```

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[알림] admin` 과 주석 세 줄 |

**💡 힌트**

1. 반복을 여는 줄이 **트리거**입니다 — 무엇이 시작을 알렸나.
2. `if` 줄이 **조건**입니다 — 어떤 경우에만 움직이나.
3. 안쪽 줄이 **액션**입니다 — 그래서 무엇을 하나.


In [ ]:
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

# 줄 끝의 화살표 뒤에 트리거 · 조건 · 액션 중 무엇인지 적으세요
for event in events:                       # 1. →
    if event["rule"] == "brute_force":     # 2. →
        print("[알림]", event["user"])      # 3. →


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 5-4 · 조건만 바꿔 보기</font></h3></td></tr></table>

위 코드에서 **조건 줄 하나만** 바꿔 `night_login` alert(경보)가 나오게 하시오. 나머지 줄은 손대지 않습니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[알림] admin` — 이번에는 심야 접속 때문 |

**💡 힌트**

1. `if` 줄의 값만 바꿉니다.
2. 트리거와 액션은 그대로 둡니다.
3. **갈라 적으면 한 줄만 고치면 된다**는 것이 이 문제의 요점입니다.


In [ ]:
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

# 조건 줄 하나만 고쳐 night_login 경보가 나오게 하세요. 나머지 두 줄은 그대로 둡니다
for event in events:
    if event["rule"] == "brute_force":
        print("[알림]", event["user"])


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.2 액션을 함수로 — 웹훅 수신 서버로 전송한다</mark>

액션을 함수로 떼어 두면 **알림 방식을 바꿀 때 한 곳만** 고칩니다.
오늘은 그 액션이 **오전에 만든 웹훅 서버로 보내는 것**입니다.

```python
import requests


def send_alert(event):
    response = requests.post("http://127.0.0.1:5005/webhook", json=event, timeout=5)
    return response.json()
```

- `requests.post` 는 9/30에 배운 `get` 의 짝입니다. **보낼 내용은 `json=` 에** 담습니다.
- 준비 셀이 웹훅 수신 서버를 **5005** 번 포트에서 실행해 두었습니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-5 · 무엇이 보일까요</font></h3></td></tr></table>

서버가 살아 있는지 먼저 봅니다.

```bash
curl -s -o /dev/null -w "%{http_code}" -X POST -H "Content-Type: application/json" -d '{"rule":"ping"}' http://127.0.0.1:5005/webhook
```

막히면 바로 위 `5.2 액션을 함수로` 설명을 다시 봅니다.


💻 **터미널에 입력합니다** — 노트북 셀이 아닙니다.

```bash
curl -s -o /dev/null -w "%{http_code}" -X POST -H "Content-Type: application/json" -d '{"rule":"ping"}' http://127.0.0.1:5005/webhook
```

✅ `200`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-6 · 무엇이 보일까요</font></h3></td></tr></table>

이번에는 파이썬에서 보냅니다.

```python
import requests

response = requests.post("http://127.0.0.1:5005/webhook",
                         json={"rule": "brute_force", "user": "admin"}, timeout=5)

print(response.json()["status"])
```

막히면 바로 위 `5.2 액션을 함수로` 설명을 다시 봅니다.


In [ ]:
import requests

response = requests.post("http://127.0.0.1:5005/webhook",
                         json={"rule": "brute_force", "user": "admin"}, timeout=5)

print(response.json()["status"])


✅ `ok`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-7 · `send_alert` 만들기</font></h3></td></tr></table>

alert(경보) 하나를 받아 서버로 보내는 **`send_alert`** 함수를 만드시오. 서버가 돌려준 `count` 를 돌려줍니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 숫자 하나 (보낼 때마다 늘어난다) |

**💡 힌트**

1. `def send_alert(event):` 로 시작합니다.
2. `requests.post(주소, json=event, timeout=5)` 로 보냅니다.
3. 돌려받은 것에서 `count` 를 꺼내 `return` 합니다.


In [ ]:
import requests

URL = "http://127.0.0.1:5005/webhook"


def send_alert(event):
    # 1. requests.post 로 URL 에 event 를 보내고 response 라는 변수에 담으세요 (json=event, timeout=5)

    # 2. response.json() 에서 "count" 값을 꺼내 return 하세요



print(send_alert({"rule": "brute_force", "user": "admin"}))


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-8 · 세 건을 모두 보내기</font></h3></td></tr></table>

`events` 세 건을 모두 보내고, 보낼 때마다 **룰 이름과 결과**를 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[전송] brute_force → ok` 꼴로 세 줄 |

**💡 힌트**

1. 문제 5-7의 함수를 그대로 씁니다. 이번에는 `status` 를 돌려주게 해도 됩니다.
2. `for` 로 세 건을 돕니다.
3. f-string 으로 한 줄씩 냅니다.


In [ ]:
import requests

events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

URL = "http://127.0.0.1:5005/webhook"


def send_alert(event):
    # 1. requests.post 로 URL 에 event 를 보내고 response 라는 변수에 담으세요 (json=event, timeout=5)

    # 2. response.json() 에서 "status" 값을 꺼내 return 하세요



# 3. for 로 events 에서 경보를 하나씩 꺼내 event 에 담으세요

    # 4. send_alert(event) 가 돌려준 값을 result 라는 변수에 담으세요

    # 5. f"[전송] {event['rule']} → {result}" 를 출력하세요



<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-9 · 보낸 것을 서버에서 확인하기</font></h3></td></tr></table>

서버가 남긴 **`received_alerts.json`** 을 읽어 **몇 건 받았는지**와 **룰 이름들**을 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 받은 건수와 룰 이름 목록 |

**💡 힌트**

1. 서버가 받을 때마다 그 파일에 저장합니다.
2. `json.load` 로 읽으면 리스트입니다.
3. `for` 로 돌며 `rule` 을 꺼냅니다.


In [ ]:
import json

# 1. "received_alerts.json" 을 open 으로 열고(encoding="utf-8") json.load 로 읽어 rows 라는 변수에 담으세요


# 2. f"받은 경보 {len(rows)}건" 을 출력하세요

# 3. for 로 rows 를 돌면서 rule 값을 한 줄씩 출력하세요




<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 5-10 · 실패해도 멈추지 않게</font></h3></td></tr></table>

서버가 꺼져 있을 때도 프로그램이 죽지 않게 `send_alert` 를 고치시오. 9/30에 배운 그대로입니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 꺼진 포트로 보내면 `전송 실패` 가 출력되고 다음 줄이 이어진다 |

**💡 힌트**

1. `try` 안에 `requests.post` 를 둡니다.
2. `except requests.RequestException:` 으로 잡습니다.
3. 실패하면 `None` 을 돌려주고, 부르는 쪽이 `if` 로 확인합니다.


In [ ]:
import requests


def send_alert(event, port=5005):
    # 1. try: 안에서 requests.post 로 f"http://127.0.0.1:{port}/webhook" 에 event 를 보내고,
    #    response.json() 의 "status" 값을 return 하세요



    # 2. except requests.RequestException: 으로 잡아 None 을 return 하세요



result = send_alert({"rule": "brute_force"}, port=59999)   # 아무도 안 듣는 포트

# 3. result 에 값이 있으면 "전송 성공" 을, 없으면 "전송 실패" 를 출력하세요




print("프로그램은 살아 있습니다")


### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.정리 📋 한눈에</mark>

| 조각 | 코드에서 어디 |
|---|---|
| 트리거 | 반복을 여는 줄 — 무엇이 시작을 알렸나 |
| 조건 | `if` 줄 — 어떤 경우에만 움직이나 |
| 액션 | 안쪽 줄 — 그래서 무엇을 하나 |

| 쓰는 법 | 뜻 |
|---|---|
| `requests.post(주소, json=값)` | 보낼 내용을 본문에 담아 보낸다 |
| 액션을 함수로 | 알림 방식을 바꿀 때 **한 곳만** 고친다 |


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">6교시 (15:00–15:50) · 같은 알림이 세 번 가면</mark>


## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다. 검색이든 공식 문서든 AI 든 상관없습니다. 찾은 것은 노트북 셀이나 터미널에서 한 번 실행해 확인합니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **멱등성(idempotency)** | 같은 일을 두 번 해도 결과가 같다는 것이 왜 중요한가 |
| **중복 알림** | 실무에서 알림이 반복되면 어떤 일이 생기나 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다. 한 줄씩이면 됩니다.

- 멱등성(idempotency) →
- 중복 알림 →

적은 내용은 노트북 파일에 함께 저장됩니다.

적어 둔 것은 이 교시가 끝날 때 다시 봅니다. 찾은 것과 배운 것이 어디서 갈렸는지가 오늘의 공부입니다.


---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">이미 보낸 것은 다시 보내지 않는다</mark>


### 왜 필요한가

1. 스케줄러가 5분마다 로그를 읽는다고 해 봅시다. 그런데 **같은 로그가 파일에 그대로** 있습니다.
2. 다음 실행에서도 같은 사건을 보고 **또 알림을 보냅니다.** 하루면 288번입니다.
3. 같은 알림이 세 번 가면 **아무도 안 봅니다.** 진짜 alert(경보)가 그 사이에 묻힙니다.
4. 같은 일을 두 번 해도 결과가 한 번 한 것과 같아야 합니다. 이 성질을 **멱등성**이라고 합니다.


### 이 시간에 나오는 말

| 말 | 뜻 |
|---|---|
| 멱등성 | 같은 일을 여러 번 해도 결과가 한 번 한 것과 같은 성질 |
| 사건 번호 | 사건 하나를 가리키는 고유한 이름 |
| `processed_ids.json` | 이미 처리한 사건 번호를 적어 두는 파일 |


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">6.1 중복이 나는 장면을 먼저 본다</mark>

막는 법을 배우기 전에 **실제로 벌어지는 것**을 봅니다.

```python
for turn in [1, 2]:                  # 스케줄러가 두 번 실행되었다고 하면
    for event in events:
        print(f"{turn}회차 [전송] {event['rule']}")
```

같은 경보가 **두 번씩** 나갑니다. 5분마다면 하루에 288번입니다.


아래 셀을 먼저 실행합니다. 5교시에서 쓴 `events` 를 다시 만듭니다.


In [2]:
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

print(f"경보 {len(events)}건")


경보 3건


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 **무엇이** 보일지 적어 보세요. 적은 뒤에 실행해 맞춰 봅니다.

```python
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

sent = 0
for turn in [1, 2]:
    for event in events:
        sent = sent + 1

print("보낸 횟수:", sent)
```

막히면 바로 위 `6.1 중복이 나는 장면을 먼저 본다` 설명을 다시 봅니다.


In [5]:
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

sent = 0
for turn in [1, 2]:
    for event in events:
        sent = sent + 1

print("보낸 횟수:", sent)


보낸 횟수: 6


✅ `보낸 횟수: 6`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-2 · 두 번 실행해 보기</font></h3></td></tr></table>

스케줄러가 **두 번** 실행되었다고 하고, 보내는 줄을 화면에 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `1회차 [전송] …` 세 줄, `2회차 [전송] …` 세 줄 |

**💡 힌트**

1. 바깥 반복이 회차, 안쪽 반복이 alert(경보)입니다.
2. 반복 안에 반복을 넣습니다.
3. f-string 에 회차와 룰 이름을 함께 넣습니다.


In [ ]:
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

# 1. for 로 [1, 2] 를 돌면서 turn 이라는 변수에 담으세요
 
    # 2. 그 안에서 for 로 events 를 돌면서 event 에 담으세요

        # 3. f"{turn}회차 [전송] {event['rule']}" 를 출력하세요
         
    
       



In [8]:
for turn in [1, 2]:
    for event in events:
        print(f"{turn}회차 [전송] {event['rule']}")

1회차 [전송] brute_force
1회차 [전송] password_spraying
1회차 [전송] night_login
2회차 [전송] brute_force
2회차 [전송] password_spraying
2회차 [전송] night_login


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-3 · 이미 보낸 것을 걸러 보기</font></h3></td></tr></table>

이미 보낸 번호를 담은 리스트가 있을 때, **안 보낸 것만** 골라 출력하시오.

| | |
|---|---|
| 주어지는 값 | `done = ["brute_force:admin"]` |
| 🎯 나와야 하는 결과 | `password_spraying:…` 과 `night_login:…` 두 줄 |

**💡 힌트**

1. 없는지 보는 것은 `not in` 입니다.
2. `if event["id"] not in done:` 으로 거릅니다.
3. 이 한 줄이 중복을 막는 핵심입니다.


In [ ]:
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

done = ["brute_force:admin"]

# 1. for 로 events 에서 경보를 하나씩 꺼내 event 에 담으세요

    # 2. event 의 id 값이 done 에 없으면 (not in)

        # 3. 그 id 값을 출력하세요



In [ ]:
for event in events:
    if event["id"] not in done:
        print(event["id"])

<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 6-4 · 번호를 직접 만들어 보기</font></h3></td></tr></table>

`id` 칸이 없다고 치고, **룰 이름과 계정을 이어 붙여** 번호를 만드시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `brute_force:admin` 꼴 |

**💡 힌트**

1. f-string 으로 두 값을 콜론으로 잇습니다.
2. 계정이 없는 alert(경보)는 IP 를 씁니다 — `if "user" in event:` 로 갈라 봅니다.
3. **같은 사건이면 언제나 같은 번호**가 나와야 합니다.


In [ ]:
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

for event in events:
    # 1. event 에 "user" 칸이 있으면 f"{event['rule']}:{event['user']}" 를 event_id 라는 변수에 담으세요


    # 2. 없으면 user 대신 ip 값을 써서 event_id 에 담으세요


    # 3. event_id 를 출력하세요



### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">6.2 `processed_ids.json` — 파일에 적어 둔다</mark>

리스트에만 담아 두면 **프로그램이 꺼지는 순간 사라집니다.** 파일에 남겨야 다음 실행이 압니다.

```python
import json
import os


def load_done():
    if os.path.exists("processed_ids.json"):
        with open("processed_ids.json", encoding="utf-8") as f:
            return json.load(f)
    return []                                  # 처음이면 빈 목록
```

- `os` 는 **오늘 처음 쓰는 도구**입니다. 파이썬에 들어 있어 `import os` 만 하면 됩니다 — `json`·`re` 와 같습니다.
- `os.path.exists` 는 **파일이 있는지** 보는 명령입니다. 처음 돌 때는 파일이 없습니다.
- 저장은 9/28에 배운 `json.dump` 그대로입니다.


아래 셀을 먼저 실행합니다. 읽고 쓰는 함수 두 개를 준비합니다.


In [10]:
import json
import os


def load_done():
    if os.path.exists("processed_ids.json"):
        with open("processed_ids.json", encoding="utf-8") as f:
            return json.load(f)
    return []


def save_done(done):
    with open("processed_ids.json", "w", encoding="utf-8") as f:
        json.dump(done, f, ensure_ascii=False, indent=2)

print("준비 끝")


준비 끝


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-5 · 무엇이 보일까요</font></h3></td></tr></table>

아직 파일이 없을 때 무엇이 돌아올지 적어 보세요.

```python
import os

if os.path.exists("없는파일.json"):
    print("있다")
else:
    print("없다")
```

막히면 바로 위 `6.2 processed_ids.json — 파일에 적어 둔다` 설명을 다시 봅니다.


In [11]:
import os

if os.path.exists("없는파일.json"):
    print("있다")
else:
    print("없다")


없다


✅ `없다`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-6 · 무엇이 보일까요</font></h3></td></tr></table>

저장했다가 다시 읽어 봅니다.

```python
import json

with open("demo_ids.json", "w", encoding="utf-8") as f:
    json.dump(["a", "b"], f)

with open("demo_ids.json", encoding="utf-8") as f:
    print(json.load(f))
```

막히면 바로 위 `6.2 processed_ids.json — 파일에 적어 둔다` 설명을 다시 봅니다.


In [12]:
import json

with open("demo_ids.json", "w", encoding="utf-8") as f:
    json.dump(["a", "b"], f)

with open("demo_ids.json", encoding="utf-8") as f:
    print(json.load(f))


['a', 'b']


✅ `['a', 'b']`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-7 · 처음 한 번 보내기</font></h3></td></tr></table>

`load_done()` 으로 읽어 **안 보낸 것만** 보내고, 보낸 번호를 파일에 저장하시오. 보낸 건수를 마지막에 출력합니다.

- 실제로 보내지는 말고 화면에 출력하기만 합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `보낸 건수 3` |

**💡 힌트**

1. `done = load_done()` 으로 시작합니다.
2. `if event["id"] not in done:` 로 거르고, 보낸 뒤 `done.append` 합니다.
3. 마지막에 `save_done(done)` 을 부릅니다.


In [ ]:
import json
import os


def load_done():
    if os.path.exists("processed_ids.json"):
        with open("processed_ids.json", encoding="utf-8") as f:
            return json.load(f)
    return []


def save_done(done):
    with open("processed_ids.json", "w", encoding="utf-8") as f:
        json.dump(done, f, ensure_ascii=False, indent=2)


events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

done = load_done()
sent = 0

for event in events:
    # 1. event 의 id 값이 done 에 없으면 (not in)

        # 2. "[전송]" 과 event 의 rule 값을 출력하세요

        # 3. done 에 event 의 id 값을 append 하세요

        # 4. sent 에 1 을 더하세요


save_done(done)

print("보낸 건수", sent)


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-8 · 한 번 더 실행해 보기</font></h3></td></tr></table>

문제 6-7의 코드를 **그대로 한 번 더** 실행하시오. 이번에는 몇 건이 나가는지 봅니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `보낸 건수 0` |

**💡 힌트**

1. 코드를 고치지 않습니다. 같은 셀을 다시 실행하면 됩니다.
2. 앞 실행이 `processed_ids.json` 에 세 건을 적어 두었습니다.
3. 0 이 나오면 **멱등성이 생긴 것**입니다.


In [ ]:
# 문제 6-7 의 셀로 올라가 그대로 한 번 더 실행합니다. 이 셀에는 쓰지 않습니다


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-9 · 새 alert(경보)가 하나 더 오면</font></h3></td></tr></table>

alert 목록에 **새 사건 하나**를 더해 다시 실행하시오. 새 것만 나가야 합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `보낸 건수 1` |

**💡 힌트**

1. `events` 리스트에 딕셔너리 하나를 `append` 합니다.
2. `id` 는 앞의 것들과 달라야 합니다.
3. 나머지 코드는 문제 6-7 그대로입니다.


In [ ]:
import json
import os


def load_done():
    if os.path.exists("processed_ids.json"):
        with open("processed_ids.json", encoding="utf-8") as f:
            return json.load(f)
    return []


def save_done(done):
    with open("processed_ids.json", "w", encoding="utf-8") as f:
        json.dump(done, f, ensure_ascii=False, indent=2)


events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

# 1. events 에 새 경보 하나를 append 하세요 — id "brute_force:guest", rule "brute_force", user "guest", count 3


done = load_done()
sent = 0

for event in events:
    # 2. event 의 id 값이 done 에 없으면 (not in)

        # 3. "[전송]" 과 id 값을 출력하고, done 에 id 값을 append 하고, sent 에 1 을 더하세요




save_done(done)

print("보낸 건수", sent)


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 6-10 · 기록을 지우면</font></h3></td></tr></table>

`processed_ids.json` 을 **지우고** 다시 실행하면 어떻게 되는지 확인하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 다시 전부 나간다 — 전송 기록이 사라졌기 때문 |

**💡 힌트**

1. 파일을 지우는 명령은 `rm -f processed_ids.json` 입니다. 터미널에 입력합니다. `-f` 를 붙이면 파일이 없어도 오류가 나지 않습니다.
2. 지운 뒤 문제 6-7 을 다시 돌립니다.
3. **전송 기록이 파일에 있다**는 것이 이 문제의 요점입니다. 파일을 잃으면 중복이 다시 납니다.


💻 **터미널에 입력합니다** — 노트북 셀이 아닙니다.

```bash
rm -f processed_ids.json
```

그다음 문제 6-7 의 셀을 다시 실행하고, 나온 건수를 확인합니다.

### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">6.정리 📋 한눈에</mark>

| 쓰는 법 | 뜻 |
|---|---|
| 사건 번호(`id`) | 같은 사건이면 **언제나 같은 값**이어야 한다 |
| `if event["id"] not in done:` | 이미 보낸 것은 건너뛴다 |
| `os.path.exists(파일)` | 파일이 있는지 본다. 처음엔 없다 |
| `processed_ids.json` | 다음 실행이 읽을 **전송 기록** |

⚠ 전송 기록이 **파일에 있습니다.** 파일을 잃으면 중복이 다시 납니다.


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">7교시 (16:00–16:50) · 정해진 간격마다 자동으로 실행한다</mark>


## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다. 검색이든 공식 문서든 AI 든 상관없습니다. 찾은 것은 노트북 셀이나 터미널에서 한 번 실행해 확인합니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **스케줄러(scheduler)** | 정해진 시각에 프로그램을 실행하는 방법에는 무엇이 있나 |
| **cron** | 운영체제가 가진 정기 실행 도구. 어떻게 시각을 적나 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다. 한 줄씩이면 됩니다.

- 스케줄러(scheduler) →
- cron →

적은 내용은 노트북 파일에 함께 저장됩니다.

적어 둔 것은 이 교시가 끝날 때 다시 봅니다. 찾은 것과 배운 것이 어디서 갈렸는지가 오늘의 공부입니다.


---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">실행 버튼을 없앤다</mark>


### 왜 필요한가

1. 지금까지 만든 것은 **사람이 실행해야** 동작합니다. 새벽 3시에 누를 사람이 없습니다.
2. 방법이 둘입니다. **파이썬 프로그램 안에서 실행하는 것**(`schedule`)과 **운영체제가 실행하는 것**(`cron`).
3. 둘의 차이는 하나입니다 — `schedule` 은 **그 프로그램이 실행 중일 때만** 동작하고, `cron` 은 **프로그램이 종료되어 있어도** 운영체제가 실행합니다.


### 이 시간에 나오는 말

| 말 | 뜻 |
|---|---|
| 스케줄러 | 정해진 때에 일을 시키는 것 |
| `schedule` | 파이썬 코드로 간격을 적는 패키지 |
| `cron` | 운영체제가 가진 정기 실행 도구 |
| cron 표현식 | 분·시·일·월·요일 **다섯 칸** |


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">7.1 `schedule` — 파이썬 안에서 돈다</mark>

```python
import schedule
import time


def job():
    print("점검합니다")


schedule.every(2).seconds.do(job)      # 2초마다 job 을 부른다

for i in range(6):                     # 여섯 번 확인하는 동안만
    schedule.run_pending()
    time.sleep(1)
```

- `schedule.every(10).minutes.do(job)` 처럼 **읽기 쉽게** 적습니다.
- **`run_pending()` 을 계속 불러 줘야** 합니다. 혼자 도는 게 아닙니다.
- 실무에서는 `while True:` 로 계속 반복합니다. 여기서는 **셀이 끝나야 하니 정해진 횟수만** 반복합니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-1 · 무엇이 보일까요</font></h3></td></tr></table>

간격을 **10분**으로 두고 몇 초만 실행해 봅니다. 화면에 무엇이 보일지 적어 보세요.

```python
import schedule
import time


def job():
    print("점검합니다")


schedule.every(10).minutes.do(job)

for i in range(3):
    schedule.run_pending()
    time.sleep(1)

print("끝")
```

막히면 바로 위 `7.1 schedule — 파이썬 안에서 돈다` 설명을 다시 봅니다.


In [ ]:
import schedule
import time


def job():
    print("점검합니다")


schedule.every(10).minutes.do(job)

for i in range(3):
    schedule.run_pending()
    time.sleep(1)

print("끝")


✅ `끝 — 10분이 안 지나서 job 은 한 번도 안 불린다`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-2 · 2초마다 실행해 보기</font></h3></td></tr></table>

`job` 을 **2초마다** 부르게 하고, 6초 동안 실행해 보시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `점검합니다` 가 두세 번 (기다린 시간에 따라 다릅니다) |

**💡 힌트**

1. `schedule.every(2).seconds.do(job)` 입니다.
2. `for i in range(6):` 안에서 `run_pending()` 과 `time.sleep(1)` 을 부릅니다.
3. 2초마다 실행되므로 6초 동안 두세 번 출력됩니다.


In [ ]:
import schedule
import time

schedule.clear()   # 앞 셀에서 걸어 둔 일을 지웁니다


def job():
    print("점검합니다")


# 1. job 을 2초마다 부르게 등록하세요 — schedule.every(2).seconds.do(job)

# 2. for i in range(6): 으로 여섯 번 돌면서

    # 3. schedule.run_pending() 을 부르고

    # 4. time.sleep(1) 로 1초 쉬세요


print("끝")


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-3 · 몇 번 실행됐는지 세기</font></h3></td></tr></table>

`job` 이 **몇 번 불렸는지** 세어 마지막에 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `점검 N번 돌았습니다` |

**💡 힌트**

1. 함수 밖에 리스트 `runs` 가 있습니다.
2. `job` 이 실행될 때마다 `runs` 에 하나씩 `append` 합니다.
3. 마지막에 `len()` 으로 셉니다.


In [ ]:
import schedule
import time

schedule.clear()   # 앞 셀에서 걸어 둔 일을 지웁니다

runs = []


def job():
    # 1. runs 에 1 을 append 하세요



# 2. job 을 2초마다 부르게 등록하세요

# 3. 여섯 번 돌면서 schedule.run_pending() 을 부르고 time.sleep(1) 로 쉬세요




print(f"점검 {len(runs)}번 돌았습니다")


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-4 · 점검 일을 진짜 일로 바꾸기</font></h3></td></tr></table>

`job` 이 **`normalized_logs.json` 을 읽어 건수를 출력하게** 고치시오. 스케줄러가 부르는 것이 이제 진짜 일입니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[점검] 정규화 16건` 이 몇 번 |

**💡 힌트**

1. `job` 함수 안에서 파일을 읽습니다.
2. 9/28에 배운 `json.load` 그대로입니다.
3. 준비 셀이 그 파일을 만들어 두었습니다.


In [ ]:
import json
import schedule
import time

schedule.clear()   # 앞 셀에서 걸어 둔 일을 지웁니다


def job():
    # 1. "normalized_logs.json" 을 open 으로 열고 json.load 로 읽어 rows 라는 변수에 담으세요


    # 2. f"[점검] 정규화 {len(rows)}건" 을 출력하세요



# 3. job 을 2초마다 부르게 등록하세요


for i in range(6):
    schedule.run_pending()
    time.sleep(1)

print("끝")


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 7-5 · 간격을 바꿔 보기</font></h3></td></tr></table>

간격을 `1`·`2`·`3` 초로 바꿔 가며 6초 동안 **몇 번 도는지** 각각 세어 보시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 간격이 짧을수록 많이 돈다 |

**💡 힌트**

1. 간격 셋을 리스트에 담고 `for` 로 돕니다.
2. 간격을 바꿀 때마다 `schedule.clear()` 로 앞 설정을 지웁니다.
3. 간격이 짧을수록 실행 횟수가 늘어납니다. 2교시의 폴링과 같습니다.


In [ ]:
import schedule
import time

for every in [1, 2, 3]:
    schedule.clear()
    runs = []

    def job():
        runs.append(1)

    # 1. job 을 every 초마다 부르게 등록하세요

    # 2. 여섯 번 돌면서 schedule.run_pending() 을 부르고 time.sleep(1) 로 쉬세요



    # 3. f"{every}초마다 — {len(runs)}번 돌았습니다" 를 출력하세요



### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">7.2 `cron` 다섯 칸 · `scheduler_job.py` 조립</mark>

`schedule` 은 **그 프로그램이 실행 중이어야** 동작합니다. 창을 닫거나 컴퓨터가 꺼지면 멈춥니다.
운영체제가 실행하면 프로그램이 종료되어 있어도 정해진 시각에 실행됩니다. 그것이 **`cron`** 입니다.

cron 은 빈칸으로 나뉜 **다섯 칸**으로 시각을 적습니다. 앞에서부터 **분 · 시 · 일 · 월 · 요일**입니다.

| 표현식 | 언제 |
|---|---|
| `0 6 * * *` | 매일 오전 6시 정각 |
| `30 2 * * *` | 매일 오전 2시 30분 |
| `*/10 * * * *` | 10분마다 |

읽는 요령이 있습니다. **별표가 아닌 칸만 읽고** 나머지는 「매번」으로 읽습니다.

> ⚠ **`cron` 은 리눅스와 맥에 들어 있는 프로그램입니다.** 윈도우에는 없습니다(윈도우는 「작업 스케줄러」를 씁니다).
> 오늘은 표현식을 **읽는 것까지만** 합니다. 위 세 가지 모양만 읽을 수 있으면 됩니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-6 · 무엇을 뜻할까요</font></h3></td></tr></table>

아래 세 표현식이 **언제**를 뜻할지 적어 보세요. 실행하는 문제가 아닙니다 — 읽는 문제입니다.

```
0 9 * * *
30 7 * * *
*/5 * * * *
```

**읽는 요령** — 별표가 아닌 칸만 읽고 나머지는 「매번」으로 읽습니다.

막히면 바로 위 `7.2 cron 다섯 칸` 설명의 표를 다시 봅니다.


✅ `매일 오전 9시 · 매일 오전 7시 30분 · 5분마다`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-7 · (선택) 표현식을 읽어 설명하기</font></h3></td></tr></table>

**선택 문제입니다.** 시간이 부족하면 건너뛰고 문제 7-9 로 갑니다.

표현식 셋을 받아 **분과 시가 무엇인지** 한 줄씩 출력하시오.

| | |
|---|---|
| 주어지는 값 | `exprs = ["0 6 * * *", "30 2 * * *", "*/5 * * * *"]` |
| 🎯 나와야 하는 결과 | `0 6 * * * → 분 0 시 6` 꼴로 세 줄 |

**💡 힌트**

1. `split()` 으로 다섯 칸을 나눕니다.
2. 0번이 분, 1번이 시입니다.
3. f-string 으로 원본과 함께 냅니다.


In [ ]:
exprs = ["0 6 * * *", "30 2 * * *", "*/5 * * * *"]

# 1. for 로 exprs 에서 표현식을 하나씩 꺼내 expr 에 담으세요

    # 2. expr.split() 으로 나눈 것을 parts 라는 변수에 담으세요

    # 3. f"{expr} → 분 {parts[0]} 시 {parts[1]}" 을 출력하세요



<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-8 · (선택) 표현식을 직접 써 보기</font></h3></td></tr></table>

**선택 문제입니다.** 시간이 부족하면 건너뛰고 문제 7-9 로 갑니다.

아래 셋을 cron 표현식으로 적으시오. 주석으로 답을 적으면 됩니다.

1. 매일 새벽 3시 정각
2. 1분마다
3. 매주 금요일 오후 6시

| | |
|---|---|
| 🎯 나와야 하는 결과 | `0 3 * * *` · `* * * * *` · `0 18 * * 5` |

**💡 힌트**

1. 칸 순서는 분·시·일·월·요일입니다.
2. 「매번」은 별표입니다.
3. 요일은 0 이 일요일입니다. 금요일은 5 입니다.


In [ ]:
# 여기에 답을 적습니다
# 1.
# 2.
# 3.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-9 · `scheduler_job.py` 만들기</font></h3></td></tr></table>

오후의 산출물 **`scheduler_job.py`** 를 완성하시오. 새 문법은 없습니다. 파일의 뼈대는 채워져 있습니다.

1. 아래 셀의 번호 주석 아래에 코드를 쓰고 실행합니다. `scheduler_job.py` 파일이 만들어집니다.
2. 그 아래 **확인** 명령을 터미널에 입력합니다. `processed_ids.json` 을 지우고 파일을 **두 번** 실행합니다.

| | |
|---|---|
| 🎯 1회차 | `[전송] brute_force:admin` 과 `점검 완료 — 새 경보 1건` |
| 🎯 2회차 | `점검 완료 — 새 경보 0건` |

**💡 힌트**

1. 걸러 내는 세 줄은 문제 6-7 와 같습니다.
2. 등록은 `schedule.every(args.every).seconds.do(job)` 입니다.
3. 2회차에 `[전송]` 줄이 또 나오면 걸러 내는 줄이 빠진 것입니다.


In [14]:
%%writefile scheduler_job.py
import argparse
import json
import os
import schedule
import time

parser = argparse.ArgumentParser(description="정기 점검 작업")
parser.add_argument("--every", type=int, default=2, help="몇 초마다 돌릴지")
args = parser.parse_args()

THRESHOLD = 3


def load_done():
    if os.path.exists("processed_ids.json"):
        with open("processed_ids.json", encoding="utf-8") as f:
            return json.load(f)
    return []


def job():
    with open("normalized_logs.json", encoding="utf-8") as f:
        rows = json.load(f)

    count = {}                                # 룰 ① — 계정마다 실패 횟수를 센다 (9/29)
    for row in rows:
        if row["level"] == "WARN":
            user = row["user"]
            if user in count:
                count[user] = count[user] + 1
            else:
                count[user] = 1

    done = load_done()
    sent = 0

    for user in count:
        if count[user] >= THRESHOLD:
            event_id = f"brute_force:{user}"
            # 1. event_id 가 done 에 없으면 (not in)

                # 2. "[전송]" 과 event_id 를 출력하고, done 에 event_id 를 append 하고, sent 에 1 을 더하세요




    with open("processed_ids.json", "w", encoding="utf-8") as f:
        json.dump(done, f, ensure_ascii=False, indent=2)

    print(f"점검 완료 — 새 경보 {sent}건")


# 3. job 을 args.every 초마다 부르게 등록하세요


for i in range(4):
    schedule.run_pending()
    time.sleep(1)


Writing scheduler_job.py


아래 **확인** 명령을 터미널에 입력합니다. 6교시에 저장한 `processed_ids.json` 을 먼저 지웁니다.


💻 **확인 — 터미널에 그대로 입력합니다.**

```bash
rm -f processed_ids.json
python scheduler_job.py
python scheduler_job.py
```

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">📌 마무리 · 깃허브에 올리기</font></h3></td></tr></table>

오후 산출물 두 개를 깃허브 저장소 `security-agent-toolkit` 에 올리시오.

| | |
|---|---|
| 🎯 확인 | 깃허브 저장소의 `agent_core` 폴더에 `scheduler_job.py` 와 `processed_ids.json` 이 보인다 |

**💡 힌트**

1. `git status` 로 새로 만든 파일이 목록에 있는지 먼저 봅니다.
2. `processed_ids.json` 은 문제 7-9 의 확인 명령을 실행해야 만들어집니다.
3. 오전의 `webhook_server.py` 와 같은 폴더입니다.

💻 **터미널에 입력합니다** — 노트북 셀이 아닙니다.

```bash
cd ~/security-agent-toolkit
git status
git add agent_core/scheduler_job.py agent_core/processed_ids.json
git commit -m "Add scheduler job"
git push
cd agent_core
```

<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 7-10 · 룰 세 개를 모두 실행하기</font></h3></td></tr></table>

`scheduler_job.py` 가 **룰 ①②③ 을 모두** 실행해 새 alert(경보)만 보내게 고치시오.

| | |
|---|---|
| 🎯 1회차 | alert 세 건 |
| 🎯 2회차 | 0건 |

**💡 힌트**

1. 9/29 오후 문제 5-11 의 세 룰 코드를 `job` 안에 넣습니다.
2. alert 마다 `id` 를 만들어 `done` 과 견줍니다.
3. 룰마다 `id` 모양이 다릅니다 — 계정·IP·시각 중 무엇을 쓸지 정합니다.


In [ ]:
# 문제 7-9 의 scheduler_job.py 셀을 이 셀에 복사해 온 뒤, job 안에 룰 ② 와 룰 ③ 을 더합니다


### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">7.정리 📋 한눈에</mark>

| 쓰는 법 | 뜻 |
|---|---|
| `schedule.every(10).minutes.do(job)` | `job` 을 10분마다 실행하도록 등록한다 |
| `schedule.run_pending()` | 간격이 지났으면 작업을 실행한다. **반복해서 호출**한다 |
| `cron` 다섯 칸 | 분 · 시 · 일 · 월 · 요일 |
| `0 6 * * *` | 매일 오전 6시 |

| | `schedule` | `cron` |
|---|---|---|
| 작업을 실행하는 주체 | 실행 중인 파이썬 프로그램 | 운영체제 |
| 프로그램을 종료하면 | 작업이 멈춘다 | 정해진 시각에 실행한다 |
| 윈도우에서 | 된다 | **안 된다** (윈도우에는 없다) |

오후의 산출물은 `agent_core` 폴더의 **`scheduler_job.py`** 와 **`processed_ids.json`** 입니다.


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">8교시 (17:00–17:30) · 마무리</mark>


새로 배우는 것이 없습니다. 오늘 만든 것을 확인하고 마무리합니다.

### 8.1 오늘 만든 것 네 개

| 파일 | 무엇 | 언제 |
|---|---|---|
| `webhook_server.py` | alert(경보)를 받는 웹훅 수신 서버 | 오전 4교시 |
| `test_webhook.sh` | 서버에 요청을 보내 확인하는 셸 스크립트 | 오전 3교시 |
| `scheduler_job.py` | 정해진 간격마다 새 alert 만 전송한다 | 오후 7교시 |
| `processed_ids.json` | 이미 전송한 사건 번호를 저장한 파일 | 오후 6교시 |

### 8.2 오늘 못 한 것 — `cron`

`cron` 은 리눅스와 맥에 들어 있는 프로그램입니다. **윈도우 PC 에는 없습니다.**
표현식을 읽는 것까지 했습니다. 리눅스 서버에서 운영할 때 이 표현식으로 등록합니다.

### 8.3 지금까지 배운 것

| 날 | 무엇을 더했나 |
|---|---|
| 9/22 · 9/23 | 값을 담고 파일을 읽는다 |
| 9/28 | 깨져도 멈추지 않고 기록한다 |
| 9/29 | 줄글에서 꺼내 수상한 것을 고른다 |
| 9/30 | 밖에 물어본다 |
| **10/2** | **요청을 받는 서버를 만들고 정해진 간격마다 실행한다** |

### 8.4 다음 주 예고

10/6 은 **LLM 과 에이전트**입니다. 오늘 만든 **트리거 · 조건 · 액션** 세 칸에서
**조건** 자리를 사람이 적은 규칙 대신 **모델이 판단**하게 바꾸는 것이 그날의 이야기입니다.


---

# <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">정답 · 먼저 풀어 본 뒤에 엽니다</mark>

정답은 아래에 그대로 있습니다. 먼저 스스로 풀어 본 뒤에 봅니다.


In [ ]:
# 정답 5-2
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

for event in events:
    if event["rule"] != "night_login":
        print(event["rule"])


In [ ]:
# 정답 5-3
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

for event in events:                          # 트리거 — 경보 목록이 있다
    if event["rule"] == "brute_force":        # 조건 — 브루트포스인가
        print("[알림]", event["user"])         # 액션 — 알린다


In [ ]:
# 정답 ⭐5-4
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

for event in events:
    if event["rule"] == "night_login":
        print("[알림]", event["user"])


In [ ]:
# 정답 5-7
import requests

URL = "http://127.0.0.1:5005/webhook"


def send_alert(event):
    response = requests.post(URL, json=event, timeout=5)
    return response.json()["count"]


print(send_alert({"rule": "brute_force", "user": "admin"}))


In [ ]:
# 정답 5-8
import requests

events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

URL = "http://127.0.0.1:5005/webhook"


def send_alert(event):
    response = requests.post(URL, json=event, timeout=5)
    return response.json()["status"]


for event in events:
    result = send_alert(event)
    print(f"[전송] {event['rule']} → {result}")


In [ ]:
# 정답 5-9
import json

with open("received_alerts.json", encoding="utf-8") as f:
    rows = json.load(f)

print(f"받은 경보 {len(rows)}건")

for row in rows:
    print(row["rule"])


In [ ]:
# 정답 ⭐5-10
import requests


def send_alert(event, port=5005):
    try:
        response = requests.post(f"http://127.0.0.1:{port}/webhook", json=event, timeout=5)
        return response.json()["status"]
    except requests.RequestException:
        return None


result = send_alert({"rule": "brute_force"}, port=59999)   # 아무도 안 듣는 포트

if result:
    print("전송 성공:", result)
else:
    print("전송 실패 — 다음 줄은 그대로 이어집니다")

print("프로그램은 살아 있습니다")


In [ ]:
# 정답 6-2
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

for turn in [1, 2]:
    for event in events:
        print(f"{turn}회차 [전송] {event['rule']}")


In [ ]:
# 정답 6-3
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

done = ["brute_force:admin"]

for event in events:
    if event["id"] not in done:
        print(event["id"])


In [ ]:
# 정답 ⭐6-4
events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

for event in events:
    if "user" in event:
        event_id = f"{event['rule']}:{event['user']}"
    else:
        event_id = f"{event['rule']}:{event['ip']}"
    print(event_id)


In [ ]:
# 정답 6-7
import json
import os


def load_done():
    if os.path.exists("processed_ids.json"):
        with open("processed_ids.json", encoding="utf-8") as f:
            return json.load(f)
    return []


def save_done(done):
    with open("processed_ids.json", "w", encoding="utf-8") as f:
        json.dump(done, f, ensure_ascii=False, indent=2)

events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

done = load_done()
sent = 0

for event in events:
    if event["id"] not in done:
        print("[전송]", event["rule"])
        done.append(event["id"])
        sent = sent + 1

save_done(done)

print("보낸 건수", sent)


**정답 6-8**

문제 6-7 의 셀을 다시 실행하면 됩니다.

앞 실행이 processed_ids.json 에 세 건을 적어 두었으므로 이번에는 0 건입니다.

In [ ]:
# 정답 6-9
import json
import os


def load_done():
    if os.path.exists("processed_ids.json"):
        with open("processed_ids.json", encoding="utf-8") as f:
            return json.load(f)
    return []


def save_done(done):
    with open("processed_ids.json", "w", encoding="utf-8") as f:
        json.dump(done, f, ensure_ascii=False, indent=2)

events = [
    {"id": "brute_force:admin", "rule": "brute_force", "user": "admin", "count": 4},
    {"id": "password_spraying:185.220.101.34", "rule": "password_spraying", "ip": "185.220.101.34", "accounts": 4},
    {"id": "night_login:admin:03:17:09", "rule": "night_login", "user": "admin", "time": "03:17:09"},
]

events.append({"id": "brute_force:guest", "rule": "brute_force", "user": "guest", "count": 3})

done = load_done()
sent = 0

for event in events:
    if event["id"] not in done:
        print("[전송]", event["id"])
        done.append(event["id"])
        sent = sent + 1

save_done(done)

print("보낸 건수", sent)


**정답 ⭐6-10**

💻 터미널에 입력합니다.

```bash
rm -f processed_ids.json     # 전송 기록 파일을 지운다
```

그다음 문제 6-7 을 다시 실행하면 세 건이 모두 다시 나갑니다.

멱등성은 코드가 아니라 이 파일이 지키고 있습니다.

In [ ]:
# 정답 7-2
import schedule
import time

schedule.clear()


def job():
    print("점검합니다")


schedule.every(2).seconds.do(job)

for i in range(6):
    schedule.run_pending()
    time.sleep(1)

print("끝")


In [ ]:
# 정답 7-3
import schedule
import time

schedule.clear()

runs = []


def job():
    runs.append(1)


schedule.every(2).seconds.do(job)

for i in range(6):
    schedule.run_pending()
    time.sleep(1)

print(f"점검 {len(runs)}번 돌았습니다")


In [ ]:
# 정답 7-4
import json
import schedule
import time

schedule.clear()


def job():
    with open("normalized_logs.json", encoding="utf-8") as f:
        rows = json.load(f)
    print(f"[점검] 정규화 {len(rows)}건")


schedule.every(2).seconds.do(job)

for i in range(6):
    schedule.run_pending()
    time.sleep(1)

print("끝")


In [ ]:
# 정답 ⭐7-5
import schedule
import time

for every in [1, 2, 3]:
    schedule.clear()
    runs = []

    def job():
        runs.append(1)

    schedule.every(every).seconds.do(job)

    for i in range(6):
        schedule.run_pending()
        time.sleep(1)

    print(f"{every}초마다 — {len(runs)}번 돌았습니다")


In [ ]:
# 정답 7-7
exprs = ["0 6 * * *", "30 2 * * *", "*/5 * * * *"]

for expr in exprs:
    parts = expr.split()
    print(f"{expr} → 분 {parts[0]} 시 {parts[1]}")


**정답 7-8**

```
1. 0 3 * * *      매일 새벽 3시 정각
2. * * * * *      1분마다
3. 0 18 * * 5     매주 금요일 오후 6시
```

In [ ]:
# 정답 7-9
code = """import argparse
import json
import os
import schedule
import time

parser = argparse.ArgumentParser(description="정기 점검 작업")
parser.add_argument("--every", type=int, default=2, help="몇 초마다 돌릴지")
args = parser.parse_args()

THRESHOLD = 3


def load_done():
    if os.path.exists("processed_ids.json"):
        with open("processed_ids.json", encoding="utf-8") as f:
            return json.load(f)
    return []


def job():
    with open("normalized_logs.json", encoding="utf-8") as f:
        rows = json.load(f)

    count = {}
    for row in rows:
        if row["level"] == "WARN":
            user = row["user"]
            if user in count:
                count[user] = count[user] + 1
            else:
                count[user] = 1

    done = load_done()
    sent = 0

    for user in count:
        if count[user] >= THRESHOLD:
            event_id = f"brute_force:{user}"
            if event_id not in done:
                print("[전송]", event_id)
                done.append(event_id)
                sent = sent + 1

    with open("processed_ids.json", "w", encoding="utf-8") as f:
        json.dump(done, f, ensure_ascii=False, indent=2)

    print(f"점검 완료 — 새 경보 {sent}건")


schedule.every(args.every).seconds.do(job)

for i in range(4):
    schedule.run_pending()
    time.sleep(1)
"""

with open("scheduler_job.py", "w", encoding="utf-8") as f:
    f.write(code)

import os

if os.path.exists("processed_ids.json"):
    os.remove("processed_ids.json")
    print("processed_ids.json 을 비웠습니다 — 6교시 기록이 남아 있으면 1회차가 0건이 됩니다")

print("scheduler_job.py 를 만들었습니다. 터미널에서 python scheduler_job.py 를 두 번 실행해 보세요.")


**마무리 · 깃허브에 올리기**

💻 터미널에 입력합니다.

```bash
cd ~/security-agent-toolkit
git status
git add agent_core/scheduler_job.py agent_core/processed_ids.json
git commit -m "Add scheduler job"
git push
cd agent_core
```

**정답 ⭐7-10**

job() 안에서 세 룰을 차례로 실행하고, 경보마다 id 를 만들어 걸러 냅니다.

```
룰 ① → f"brute_force:{user}"
룰 ② → f"password_spraying:{ip}"
룰 ③ → f"night_login:{user}:{time}"
```

id 를 만드는 규칙만 정하면 나머지는 문제 6-7 과 같습니다.